In [1]:
# notebook: 06_C2_diagnosis.ipynb
# ============================================================================
# CMVTS Extension — Diagnosing the C2 (Spearman) sign flip
# ----------------------------------------------------------------------------
# Symptom (notebook 05): C2=|Spearman(Korea-vector, target-vector)| correlates
# POSITIVELY with realized divergence (+0.80), i.e. inverted. Thailand (should be
# most similar to Korea) gets the LOWEST C2 (0.043). We isolate the cause:
#   H_a  normalization artifact  -> compute C2 on RAW vs normalized, compare
#   H_b  Korea-extremum rank collapse -> Korea tops nearly every indicator, so
#        within-country indicator ranks carry little contrast; test by removing
#        the level indicators / using country-wise ranks instead
#   H_c  original C2 definition is fragile in multi-country -> compare the paper's
#        "within-pair indicator-rank" formulation vs a "cross-country rank" one
# ============================================================================

import numpy as np
import pandas as pd
from scipy import stats

FINDEX_CSV = "findex_microdata_2025_labelled_update112425.csv"
WDI_LATEST = "wdi_macro_latest.csv"
SOURCE = "Korea, Rep."
ORDER  = ["Korea, Rep.","Indonesia","Thailand","Viet Nam","Philippines",
          "Bangladesh","Cambodia","Nepal","Pakistan","Lao PDR"]
DROP_WDI = ["D5_adult_literacy_pct"]
LAO_D4_FILL = 55.0
FINDEX_PRED = ["account","account_fin","account_mob","saved","borrowed","receive_wages"]
YES = 1

# ---- rebuild the predictor matrix (latest) ----
fx = pd.read_csv(FINDEX_CSV, low_memory=False)
def wshare(g,v): w=g["wgt"]; return 100*w[g[v]==YES].sum()/w.sum()
fx_pred = pd.DataFrame(
    [{"economy":e, **{f"FX_{v}":wshare(fx[fx.economy==e],v) for v in FINDEX_PRED}} for e in ORDER]
).set_index("economy").reindex(ORDER)

wdi = pd.read_csv(WDI_LATEST, index_col=0).reindex(ORDER).drop(columns=DROP_WDI, errors="ignore")
if pd.isna(wdi.loc["Lao PDR","D4_domestic_credit_priv_gdp"]):
    wdi.loc["Lao PDR","D4_domestic_credit_priv_gdp"] = LAO_D4_FILL
M = wdi.join(fx_pred)

# ---- outcome (Branch-2 penetration JSD) ----
def jsd(p,q,eps=1e-12):
    p=np.asarray(p,float)+eps;q=np.asarray(q,float)+eps;p/=p.sum();q/=q.sum();m=.5*(p+q)
    kl=lambda a,b:np.sum(a*np.log2(a/b));return .5*kl(p,m)+.5*kl(q,m)
p_active_src=0.581; src=np.array([1-p_active_src,p_active_src])
Y = pd.Series({e: jsd(src, np.array([1-wshare(fx[fx.economy==e],"fin8")/100,
                                     wshare(fx[fx.economy==e],"fin8")/100]))
               for e in ORDER if e!=SOURCE}, name="Y")

def corr_report(x, y, tag):
    common = x.notna() & y.notna()
    r,p = stats.pearsonr(x[common], y[common])
    rs,ps = stats.spearmanr(x[common], y[common])
    print(f"  [{tag}] Pearson {r:+.3f}(p={p:.3f}) | Spearman {rs:+.3f}(p={ps:.3f})")
    return rs

# ============================================================================
# H_a — normalization artifact?
# ============================================================================
print("="*70); print("H_a — RAW vs normalized C2 (Spearman is rank-based; "
                      "normalization should NOT change it)"); print("="*70)
def C2_from_matrix(m):
    kr = m.loc[SOURCE]
    out={}
    for e in ORDER:
        if e==SOURCE: continue
        t=m.loc[e]; c=kr.notna()&t.notna()
        rho,_=stats.spearmanr(kr[c].values,t[c].values)
        out[e]=abs(rho)
    return pd.Series(out,name="C2")

def minmax(m):
    m=m.copy(); 
    if "D1_gni_pc_atlas" in m: m["D1_gni_pc_atlas"]=np.log(m["D1_gni_pc_atlas"])
    return (m-m.min())/(m.max()-m.min())

C2_raw  = C2_from_matrix(M)
C2_norm = C2_from_matrix(minmax(M))
print("Per-country C2 raw vs normalized:")
print(pd.DataFrame({"C2_raw":C2_raw,"C2_norm":C2_norm}).round(4).to_string())
print("Diff (should be ~0 if H_a false):", np.round((C2_raw-C2_norm).abs().max(),4))
corr_report(C2_raw, Y, "C2_raw vs Y")
corr_report(C2_norm, Y, "C2_norm vs Y")

# ============================================================================
# H_b — Korea-extremum: does Korea top almost every indicator?
# ============================================================================
print("\n"+"="*70); print("H_b — Korea rank position per indicator "
                           "(1 = highest of 10). If mostly 1 -> rank collapse"); print("="*70)
ranks = M.rank(ascending=False)  # per-indicator country ranking
kr_ranks = ranks.loc[SOURCE].sort_values()
print(kr_ranks.to_string())
print(f"Korea is #1 in {(kr_ranks==1).sum()}/{M.shape[1]} indicators; "
      f"median rank {kr_ranks.median():.1f}")

# Test: drop indicators where Korea is extreme (#1 or #10), recompute C2
non_extreme = [c for c in M.columns if 1 < ranks.loc[SOURCE,c] < len(ORDER)]
print(f"Non-extreme indicators for Korea ({len(non_extreme)}):", non_extreme)
if len(non_extreme) >= 4:
    C2_ne = C2_from_matrix(M[non_extreme])
    print("C2 on non-extreme indicators:")
    print(C2_ne.round(4).to_string())
    corr_report(C2_ne, Y, "C2_nonextreme vs Y")
else:
    print("  [skip] too few non-extreme indicators")

# ============================================================================
# H_c — definition: within-pair indicator-rank vs CROSS-COUNTRY rank
# ============================================================================
print("\n"+"="*70); print("H_c — alternative C2: cross-country rank alignment"); print("="*70)
# Cross-country formulation: for each indicator, rank all 10 countries; C2 =
# 1 - normalized |rank(Korea) - rank(target)| averaged over indicators.
# This measures 'does the target sit near Korea in the cross-country ordering?'
Rc = M.rank(ascending=False)  # 1=top
def C2_crosscountry(Rc):
    kr = Rc.loc[SOURCE]; n=len(ORDER)
    out={}
    for e in ORDER:
        if e==SOURCE: continue
        t=Rc.loc[e]; c=kr.notna()&t.notna()
        # closeness in cross-country rank, averaged; 1 = identical position
        gap = (kr[c]-t[c]).abs()/(n-1)
        out[e]=1-gap.mean()
    return pd.Series(out,name="C2_cc")
C2_cc = C2_crosscountry(Rc)
print("Cross-country C2 (1 = target sits where Korea sits across indicators):")
print(C2_cc.round(4).to_string())
corr_report(C2_cc, Y, "C2_crosscountry vs Y")

# ============================================================================
# Cross-check: C3 (cosine) for reference + summary
# ============================================================================
print("\n"+"="*70); print("Reference — C3 cosine (known good) and summary"); print("="*70)
N = minmax(M); kr=N.loc[SOURCE]
C3 = pd.Series({e: float(np.dot(kr[N.loc[e].notna()&kr.notna()].values,
                                N.loc[e][N.loc[e].notna()&kr.notna()].values)/
                        (np.linalg.norm(kr[N.loc[e].notna()&kr.notna()].values)*
                         np.linalg.norm(N.loc[e][N.loc[e].notna()&kr.notna()].values)))
               for e in ORDER if e!=SOURCE}, name="C3")
corr_report(C3, Y, "C3_cosine vs Y")

summary = pd.DataFrame({"C2_raw":C2_raw,"C2_cc":C2_cc,"C3":C3,"Y":Y}).round(4)
print("\nSUMMARY (sorted by Y):")
print(summary.sort_values("Y").to_string())
print("\nInterpretation guide:")
print("  - If C2_raw==C2_norm  -> H_a false (not a normalization artifact)")
print("  - If Korea #1 in most -> H_b supported (rank collapse from extremum)")
print("  - If C2_cc flips to NEGATIVE & significant -> H_c: the cross-country")
print("    definition is the correct C2 for multi-country; paper C2 def is fragile")

H_a — RAW vs normalized C2 (Spearman is rank-based; normalization should NOT change it)
Per-country C2 raw vs normalized:
             C2_raw  C2_norm
Indonesia    0.6283   0.4669
Thailand     0.8909   0.0431
Viet Nam     0.6549   0.3990
Philippines  0.4808   0.4346
Bangladesh   0.5900   0.4705
Cambodia     0.7109   0.6214
Nepal        0.6519   0.3566
Pakistan     0.4808   0.8361
Lao PDR      0.6313   0.4655
Diff (should be ~0 if H_a false): 0.8478
  [C2_raw vs Y] Pearson -0.757(p=0.018) | Spearman -0.753(p=0.019)
  [C2_norm vs Y] Pearson +0.637(p=0.065) | Spearman +0.800(p=0.010)

H_b — Korea rank position per indicator (1 = highest of 10). If mostly 1 -> rank collapse
A1_internet_use_pct             1.0
A2_mobile_subs_p100             1.0
A3_fixed_bbnd_p100              1.0
A6_secure_servers_p1m           1.0
B8_atm_p100k                    1.0
D1_gni_pc_atlas                 1.0
D2_urban_pct                    1.0
D4_domestic_credit_priv_gdp     1.0
FX_account_fin                  1